# Analyse des données Agreste SAA 2023-2024 — Rendements agricoles

## Source
**Agreste** — Service de la Statistique et de la Prospective (SSP) du ministère de l'Agriculture  
**SAA** = Statistiques Agricoles Annuelles  
Disponible sur : **agreste.agriculture.gouv.fr**  
Licence : open data (réutilisation libre)

## Particularité de ce dataset
Contrairement aux autres sources (RPG, SGDBE, PAC), Agreste ne fournit **pas de données par parcelle**.  
Les rendements sont des **moyennes nationales** par culture et par année.  
On applique ensuite un **coefficient régional HdF** pour tenir compte de la supériorité agronomique des sols limoneux du Nord.  

## Rôle dans le modèle
Les rendements Agreste servent uniquement à calculer `revenu_net_ha` :  
`revenu_net_ha = (rendement_hdf × prix_tonne) + PAC_total - charges_intrants`  
C'est la **variable cible du régresseur** — pas du classificateur.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Imports                                     ║
# ╚══════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#fafafa'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

CULTURES_MODELE = ['ble_tendre','colza','betterave','pomme_de_terre',
                   'lin_fibre','pois_proteine','orge','mais_grain']
NOMS = {
    'ble_tendre':'Blé tendre','colza':'Colza','betterave':'Betterave',
    'pomme_de_terre':'Pomme de terre','lin_fibre':'Lin fibre',
    'pois_proteine':'Pois protéagineux','orge':'Orge','mais_grain':'Maïs grain',
}
COULEURS = {
    'ble_tendre':'#f59e0b','mais_grain':'#f97316','orge':'#84cc16',
    'betterave':'#8b5cf6','pomme_de_terre':'#d97706','colza':'#eab308',
    'lin_fibre':'#3b82f6','pois_proteine':'#22c55e',
}
print('✓ Imports OK')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Upload du fichier                           ║
# ╚══════════════════════════════════════════════════════════╝

from google.colab import files
print('Uploader : rendements_agreste_2024.csv')
uploaded = files.upload()
CSV_FILE = list(uploaded.keys())[0]
print(f'✓ Fichier reçu : {CSV_FILE}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Exploration du fichier brut                 ║
# ║                                                          ║
# ║  On explore la structure du fichier Agreste SAA pour     ║
# ║  comprendre son organisation avant tout traitement.      ║
# ╚══════════════════════════════════════════════════════════╝

df = pd.read_csv(CSV_FILE)

print('=== STRUCTURE DU FICHIER BRUT ===')
print(f'  Lignes   : {len(df):,}')
print(f'  Colonnes : {len(df.columns)}')
print()
print('Colonnes disponibles :')
for col in df.columns:
    dtype  = df[col].dtype
    n_null = df[col].isna().sum()
    n_uniq = df[col].nunique()
    exemple = df[col].dropna().iloc[0] if df[col].notna().any() else 'N/A'
    print(f'  {col:<25} type={str(dtype):<10} '
          f'null={n_null:>4}  uniques={n_uniq:>4}  ex: {exemple}')
print()
print('Aperçu complet :')
print(df.to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Explication des colonnes                    ║
# ║                                                          ║
# ║  Comprendre d'où viennent les valeurs et ce             ║
# ║  qu'elles représentent exactement.                       ║
# ╚══════════════════════════════════════════════════════════╝

EXPLICATIONS = {
    'culture_key':     'Identifiant de la culture dans notre modèle (ex: ble_tendre)',
    'culture_nom':     'Nom complet de la culture',
    'rdt_2023_tha':    'Rendement national 2023 en tonnes/hectare (source: Agreste SAA 2023)',
    'rdt_2024_tha':    'Rendement national 2024 en tonnes/hectare (source: Agreste SAA 2024)',
    'rdt_moy_tha':     'Moyenne des deux années (2023+2024)/2 — valeur utilisée dans le modèle',
    'coeff_hdf':       'Coefficient régional HdF (source: Agreste Mémento HdF 2023)',
    'rdt_hdf_tha':     'Rendement HdF estimé = rdt_moy_tha × coeff_hdf',
    'prix_eur_tonne':  'Prix moyen €/tonne (source: FranceAgriMer cotations 2023-2024)',
    'intrants_eur_ha': 'Charges intrants moyennes €/ha (source: Agreste RICA 2023)',
}

print('=== EXPLICATION DES COLONNES ===')
print()
for col in df.columns:
    expl = EXPLICATIONS.get(col, 'Colonne présente dans le fichier')
    print(f'  {col:<25} : {expl}')
print()
print('Note sur le coefficient régional HdF :')
print('  Les sols limoneux de profondeur des Hauts-de-France figurent parmi')
print('  les plus fertiles d\'Europe. Les rendements y dépassent la moyenne')
print('  nationale de 10 à 15% selon la culture.')
print('  Source : Agreste — Mémento de la statistique agricole HdF 2023')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Analyse des rendements                      ║
# ║                                                          ║
# ║  Comparaison nationale vs HdF et évolution 2023-2024.    ║
# ╚══════════════════════════════════════════════════════════╝

# S'assurer que les colonnes numériques sont bien en float
num_cols = [c for c in df.columns if 'rdt' in c or 'prix' in c or 'intrant' in c or 'coeff' in c]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print('=== RENDEMENTS NATIONAUX VS HdF ===')
print(f'{"Culture":<22} {"2023":>8} {"2024":>8} {"Moy nat."}  '
      f'{"Coeff HdF":>10} {"Rdt HdF":>9}  {"Gain HdF"}')
print('-'*82)

for _, row in df.iterrows():
    c   = row.get('culture_key', '')
    if c not in CULTURES_MODELE: continue
    r23 = row.get('rdt_2023_tha', 0)
    r24 = row.get('rdt_2024_tha', 0)
    moy = row.get('rdt_moy_tha', 0)
    cof = row.get('coeff_hdf', 1)
    hdf = row.get('rdt_hdf_tha', moy*cof)
    gain = (cof - 1) * 100
    print(f'  {NOMS[c]:<20} {r23:>8.2f} {r24:>8.2f} {moy:>9.2f}  '
          f'{cof:>9.2f}x {hdf:>9.2f}  {gain:>+6.0f}%')

print()
print('Variations 2023 → 2024 (contexte climatique) :')
for _, row in df.iterrows():
    c = row.get('culture_key', '')
    if c not in CULTURES_MODELE: continue
    r23 = row.get('rdt_2023_tha', 0)
    r24 = row.get('rdt_2024_tha', 0)
    if r23 and r23 > 0:
        var = (r24-r23)/r23*100
        signe = '▲' if var > 0 else '▼'
        print(f'  {NOMS[c]:<22} {signe} {var:+.1f}%')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Calcul du revenu net par culture            ║
# ╚══════════════════════════════════════════════════════════╝

PAC_TOTAUX = {
    'ble_tendre':180,'colza':180,'betterave':180,'pomme_de_terre':293,
    'lin_fibre':180,'pois_proteine':284,'orge':180,'mais_grain':180,
}

print('=== CALCUL DU REVENU NET PAR CULTURE ===')
print('Formule : revenu_net = (rdt_hdf × prix_tonne) + PAC_total - intrants')
print()
print(f'{"Culture":<22} {"CA brut":>9} {"PAC":>6} {"Intrants":>9} '
      f'{"Revenu net":>11}  Unité : €/ha')
print('-'*65)

revenus = {}
for _, row in df.iterrows():
    c = row.get('culture_key', '')
    if c not in CULTURES_MODELE: continue
    hdf  = row.get('rdt_hdf_tha', 0)
    prix = row.get('prix_eur_tonne', 0)
    intr = row.get('intrants_eur_ha', 0)
    pac  = PAC_TOTAUX.get(c, 180)
    ca   = hdf * prix
    rev  = ca + pac - intr
    revenus[c] = rev
    print(f'  {NOMS[c]:<20} {ca:>9,.0f} {pac:>6} {intr:>9,.0f} {rev:>10,.0f}')

print()
best = max(revenus, key=revenus.get)
worst = min(revenus, key=revenus.get)
print(f'Revenu le plus élevé  : {NOMS[best]} ({revenus[best]:,.0f} €/ha)')
print(f'Revenu le plus faible : {NOMS[worst]} ({revenus[worst]:,.0f} €/ha)')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Visualisations                              ║
# ╚══════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cols = [COULEURS[c] for c in CULTURES_MODELE]
noms = [NOMS[c] for c in CULTURES_MODELE]

# ── 1. Rendement national vs HdF ──
ax = axes[0]
df_ord = df[df['culture_key'].isin(CULTURES_MODELE)].set_index('culture_key').reindex(CULTURES_MODELE)
x = np.arange(len(CULTURES_MODELE))
w = 0.35
nat_vals = df_ord['rdt_moy_tha'].fillna(0).values
hdf_vals = df_ord['rdt_hdf_tha'].fillna(df_ord['rdt_moy_tha'] * df_ord.get('coeff_hdf', pd.Series([1]*len(CULTURES_MODELE), index=CULTURES_MODELE))).fillna(0).values
ax.bar(x-w/2, nat_vals, w, label='National', color='#cbd5e1', edgecolor='white')
ax.bar(x+w/2, hdf_vals, w, label='HdF (+coeff)', color=cols, edgecolor='white', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([NOMS[c].replace(' ','-') for c in CULTURES_MODELE],
                   rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Rendement (t/ha)')
ax.set_title('Rendement national vs HdF\n(Agreste SAA 2023-2024)', fontsize=12, pad=10)
ax.legend(fontsize=9)

# ── 2. Évolution 2023 vs 2024 ──
ax = axes[1]
r23 = df_ord['rdt_2023_tha'].fillna(0).values
r24 = df_ord['rdt_2024_tha'].fillna(0).values
ax.plot(noms, r23, 'o--', color='#94a3b8', linewidth=1.5,
        markersize=7, label='2023')
ax.plot(noms, r24, 's-', color='#1d4ed8', linewidth=2,
        markersize=8, label='2024')
ax.set_ylabel('Rendement (t/ha)')
ax.set_title('Évolution rendements\n2023 vs 2024 (national)', fontsize=12, pad=10)
ax.tick_params(axis='x', rotation=35, labelsize=9)
ax.legend(fontsize=9)

# ── 3. Revenu net par culture ──
ax = axes[2]
rev_vals = [revenus.get(c, 0) for c in CULTURES_MODELE]
bars = ax.bar(noms, rev_vals, color=cols, edgecolor='white')
ax.set_ylabel('Revenu net estimé (€/ha)')
ax.set_title('Revenu net par culture\n(rendement HdF + PAC - intrants)', fontsize=12, pad=10)
ax.tick_params(axis='x', rotation=35, labelsize=9)
ax.axhline(0, color='gray', linewidth=0.8)
for bar, v in zip(bars, rev_vals):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height() + 30 if v >= 0 else bar.get_height() - 80,
            f'{v:,.0f}€', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Analyse Agreste SAA 2023-2024 — Rendements HdF', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('agreste_rendements.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ agreste_rendements.png sauvegardé')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Synthèse et téléchargements                 ║
# ╚══════════════════════════════════════════════════════════╝

print('=' * 58)
print('  SYNTHÈSE — AGRESTE SAA 2023-2024')
print('=' * 58)
print()
print('  Source         : agreste.agriculture.gouv.fr')
print('  Données        : rendements nationaux moyens par culture')
print('  Millésimes     : SAA 2023 + SAA 2024 (moyenne)')
print()
print('  Ce qui est RÉEL :')
print('    rdt_2023_tha, rdt_2024_tha → statistiques officielles nationales')
print()
print('  Ce qui est ESTIMÉ :')
print('    coeff_hdf       → coefficient régional (Mémento Agreste HdF 2023)')
print('    rdt_hdf_tha     → rendement HdF = national × coeff')
print('    prix_eur_tonne  → prix moyen FranceAgriMer 2023-2024')
print('    intrants_eur_ha → charges moyennes Agreste RICA 2023')
print()
print('  Rôle dans le modèle :')
print('    → Calcul de revenu_net_ha (cible du régresseur uniquement)')
print('    → Non utilisé directement comme feature du classificateur')

files.download('agreste_rendements.png')
print('\n✓ Téléchargement lancé')